In [18]:
from db_config import run_query

ALLOWED = {"orgc": "wosis_orgc", 
           "ph": "wosis_phaq", 
           "clay": "wosis_clay", 
           "sand": "wosis_sand"}

def count_rows(short_name, country = "Ghana"):
    if short_name not in ALLOWED:
        raise ValueError(f"Unknown table: {short_name}")
    table = ALLOWED[short_name]
    query = f"SELECT COUNT(*) AS n FROM {table} WHERE country_name= '{country}'"
    return run_query(query)['n'].iloc[0]

def build_flat(base, others, country = "Ghana"):
    if base not in ALLOWED:
        raise ValueError("Unknown table: {base}")
    for name in others:
        if name not in ALLOWED:
            raise ValueError(f"Unknown table: {name}")

    select_parts = ['b.profile_id', 
                    'b.layer_id', 
                    f"b.value_avg AS {base}"]
    join_parts= []

    for i, name in enumerate(others, start=1):
        alias = f"t{i}"
        table = ALLOWED[name]

        select_parts.append(f"{alias}.value_avg AS {name}")
        join_parts.append(f"LEFT JOIN {table} AS {alias} ON b.layer_id = {alias}.layer_id")

        query = f"""
                SELECT {", ".join(select_parts)}
                FROM {ALLOWED[base]} AS b
                {" ".join(join_parts)}
                WHERE b.country_name = '{country}'
                 """
        df=run_query(query)

        expected = count_rows(base, country=country)
        if len(df) != expected:
            raise ValueError(
                f"Row count changed: got{len(df)}, expected {expected}")
        return df




        



In [39]:
build_flat("clay", ["sand","ph"])

,profile_id,layer_id,clay,sand
0,1414148,2909187,41.5,19.9
1,1414149,2909245,26.4,20.2
2,1422370,598041,7.1,77.1
3,1422370,598042,8.6,73.4
4,1422370,598043,8.0,73.8
...,...,...,...,...
796,1344695,2909162,22.3,25.1
797,1344696,2909225,10.7,54.2
798,1344697,2909231,16.2,48.6
799,1344698,2909242,43.0,32.7


In [24]:
build_flat("sand", ["clay","ph"])

,profile_id,layer_id,sand,clay
0,1144082,598054,56.3,11.7
1,1144082,598055,54.5,14.1
2,1144082,598056,43.0,26.0
3,1144539,189,54.5,33.3
4,1144539,190,46.3,40.8
...,...,...,...,...
829,1622420,3143835,77.0,NaN
830,1691415,3139974,14.0,NaN
831,1344108,3143831,58.0,NaN
832,1379140,3166767,26.0,NaN


In [26]:
from db_config import get_engine
engine = get_engine()

flat = build_flat("clay", ["sand","ph"])
flat.to_sql("flat_clay_ghana", engine, if_exists= 'replace', index=False)
check = run_query("SELECT * FROM flat_clay_ghana")



In [32]:
flat.shape == check.shape

True

In [37]:
flat.isnull().sum()

profile_id    0
layer_id      0
clay          0
sand          0
dtype: int64

In [38]:
build_flat("clay", ["sand"], country="Narnia")

,profile_id,layer_id,clay,sand
